# MaIA_SDCI Final Project

**Sistemas de Decision y Control Inteligente (SDCI)**  
**Universidad de los Andes, 2026**

Copyright (c) 2026 Leffer Trochez

---

Provides the graphical interface for configuring, running, and reviewing simulations. The interface uses ipywidgets so the same notebook works in VS Code/Jupyter and Google Colab.


In [ ]:
from __future__ import annotations

from io import BytesIO
import copy
import sys
import threading
import time
import traceback

import numpy as np
import ipywidgets as widgets
from IPython.display import display
from matplotlib.figure import Figure
from matplotlib import image as mpl_image
from matplotlib.patches import Rectangle


class Control_App:
    """Notebook GUI for configuring, running, and reviewing simulations."""

    MaxPlotPoints = 5000
    DefaultStopTimeMinutes = 30.0
    DefaultDisturbanceAmplitude = 13.0
    DefaultDisturbanceStartMinutes = 5.0
    DefaultDisturbanceEndMinutes = 8.0

    CONTROLLER_NAMES = {
        1: "ONOFF",
        2: "FUZZY",
        3: "MPC",
        4: "ESC",
        5: "REPLICATOR",
    }

    def __init__(self, project_context):
        self.context = project_context
        self.PROJECT = project_context["PROJECT"]
        self.CFG = project_context["CFG"]
        self.MODEL = project_context["MODEL"]
        self.ENV = project_context["ENV"]
        self.SENSOR = project_context["SENSOR"]
        self.CTRL = project_context["CTRL"]

        # Runtime mode:
        # - VS Code/Jupyter: original threaded simulation with live plots.
        # - Google Colab: synchronous simulation with one final GUI refresh.
        self._running_in_colab = self._detect_colab()
        self._colab_running = False

        self._simulation_thread = None
        self._stop_event = threading.Event()
        self._pause_event = threading.Event()
        self._data_lock = threading.Lock()
        self._latest_data = self._empty_data()
        self._last_run = None
        self._experiment_name_customized = False
        self._last_render_wall_time = 0.0
        self._plot_start_index = 0

        self._create_interface()
        self._load_configuration()
        self._configure_runtime_mode()
        self._render_plots()
        self._set_status("Ready", "ready")

    @staticmethod
    def _detect_colab():
        """Return True only when the notebook is running in Google Colab."""
        return "google.colab" in sys.modules

    def _configure_runtime_mode(self):
        """Configure controls that differ between local Jupyter and Colab."""
        if self._running_in_colab:
            # Colab does not reliably refresh ipywidgets from a background
            # simulation. Therefore pacing, pause, and stop are intentionally
            # unavailable there. The experiment is calculated as fast as
            # possible and the complete GUI is refreshed once at the end.
            self.pacing_enabled_checkbox.disabled = True
            self.pacing_enabled_checkbox.description = (
                "Enable simulation pacing (local only)"
            )
            self.pacing_slider.disabled = True
            self.pause_button.disabled = True
            self.stop_button.disabled = True
        else:
            self.pacing_enabled_checkbox.description = "Enable simulation pacing"
            self._update_pacing_controls()

    @staticmethod
    def _empty_data():
        return {
            "time": [],
            "T_ref": [],
            "T_meas": [],
            "Q_heat": [],
            "duty_cycle": [],
            "T_room": [],
            "T_dist": [],
        }

    @staticmethod
    def _section(text):
        return widgets.HTML(
            value=f"<b style='font-size:14px'>{text}</b>",
            layout=widgets.Layout(margin="7px 0 2px 0"),
        )

    @staticmethod
    def _row(label_text, widget):
        label = widgets.Label(label_text, layout=widgets.Layout(width="130px"))
        widget.layout.width = "210px"
        return widgets.HBox([label, widget], layout=widgets.Layout(width="100%"))

    def _create_interface(self):
        # EXPERIMENT
        self.experiment_name_field = widgets.Text(value="MaIA_SDCI_Run")
        self.experiment_name_field.observe(self._on_experiment_name_changed, names="value")

        # SETUP
        self.controller_dropdown = widgets.Dropdown(
            options=[
                ("ON/OFF", 1),
                ("Fuzzy Logic", 2),
                ("Model Predictive Control", 3),
                ("Extremum Seeking Control", 4),
                ("Replicator Dynamics", 5),
            ],
            value=1,
        )
        self.controller_dropdown.observe(self._on_controller_changed, names="value")

        # The Python project has one equation-based model only.
        self.model_type_dropdown = widgets.Dropdown(
            options=[("Equation-Based", 1)], value=1, disabled=True
        )

        # TEMPERATURE REFERENCES
        self.reference_title = widgets.HTML("<b>Temperature References [°C]</b>")
        self.reference_room_labels = []
        self.reference_fields = []
        self.reference_boxes = []
        for room in range(4):
            label = widgets.Label(f"Room {room + 1}", layout=widgets.Layout(width="72px"))
            field = widgets.FloatText(value=20.0, layout=widgets.Layout(width="72px"))
            field.observe(self._on_reference_changed, names="value")
            box = widgets.VBox(
                [label, field],
                layout=widgets.Layout(width="78px", align_items="center"),
            )
            self.reference_room_labels.append(label)
            self.reference_fields.append(field)
            self.reference_boxes.append(box)
        self.references_box = widgets.HBox(
            self.reference_boxes,
            layout=widgets.Layout(width="100%", justify_content="space-between"),
        )

        # ENVIRONMENT
        self.ambient_temperature_field = widgets.FloatText(value=8.0)
        self.ambient_temperature_field.observe(self._on_configuration_changed, names="value")

        self.disturbance_enabled_checkbox = widgets.Checkbox(
            value=False, description="Enable temperature disturbance", indent=False
        )
        self.disturbance_enabled_checkbox.observe(
            self._on_disturbance_enabled_changed, names="value"
        )

        self.disturbance_amplitude_field = widgets.FloatText(
            value=self.DefaultDisturbanceAmplitude
        )
        self.disturbance_amplitude_field.observe(
            self._on_configuration_changed, names="value"
        )

        self.disturbance_start_field = widgets.FloatText(
            value=self.DefaultDisturbanceStartMinutes,
            layout=widgets.Layout(width="80px"),
        )
        self.disturbance_end_field = widgets.FloatText(
            value=self.DefaultDisturbanceEndMinutes,
            layout=widgets.Layout(width="80px"),
        )
        self.disturbance_start_field.observe(self._on_configuration_changed, names="value")
        self.disturbance_end_field.observe(self._on_configuration_changed, names="value")
        disturbance_times = widgets.HBox([
            widgets.Label("Start [min]", layout=widgets.Layout(width="72px")),
            self.disturbance_start_field,
            widgets.Label("End [min]", layout=widgets.Layout(width="65px")),
            self.disturbance_end_field,
        ])

        self.disturbance_all_checkbox = widgets.Checkbox(
            value=True, description="All", indent=False, layout=widgets.Layout(width="58px")
        )
        self.disturbance_all_checkbox.observe(self._set_all_disturbance_rooms, names="value")
        self.disturbance_room_checkboxes = []
        for room in range(4):
            checkbox = widgets.Checkbox(
                value=True,
                description=str(room + 1),
                indent=False,
                layout=widgets.Layout(width="52px"),
            )
            checkbox.observe(self._on_disturbance_room_changed, names="value")
            self.disturbance_room_checkboxes.append(checkbox)
        self.affected_rooms_box = widgets.HBox(
            [self.disturbance_all_checkbox] + self.disturbance_room_checkboxes
        )

        # SIMULATION
        self.stop_time_field = widgets.FloatText(value=self.DefaultStopTimeMinutes)
        self.stop_time_field.observe(self._on_stop_time_changed, names="value")

        self.pacing_enabled_checkbox = widgets.Checkbox(
            value=True, description="Enable simulation pacing", indent=False
        )
        self.pacing_enabled_checkbox.observe(self._on_pacing_enabled_changed, names="value")

        self.pacing_slider = widgets.FloatSlider(
            value=100.0,
            min=70.0,
            max=200.0,
            step=1.0,
            description="",
            readout=True,
            continuous_update=False,
            layout=widgets.Layout(width="330px"),
        )
        self.pacing_slider.observe(self._on_configuration_changed, names="value")

        self.start_button = widgets.Button(description="START", button_style="success")
        self.pause_button = widgets.Button(description="PAUSE", button_style="warning")
        self.stop_button = widgets.Button(description="STOP", button_style="danger")
        self.clear_button = widgets.Button(description="Clear Plots")
        self.save_metrics_button = widgets.Button(description="Save Metrics")

        self.start_button.on_click(self._start_simulation)
        self.pause_button.on_click(self._pause_simulation)
        self.stop_button.on_click(self._stop_simulation)
        self.clear_button.on_click(self._clear_plots)
        self.save_metrics_button.on_click(self._save_metrics)

        self.status_html = widgets.HTML()
        self.message_html = widgets.HTML()

        left_children = [
            self._row("Experiment", self.experiment_name_field),
            self._row("Controller", self.controller_dropdown),
            self._row("Model Type", self.model_type_dropdown),
            self.reference_title,
            self.references_box,
            self._section("Environment"),
            self._row("T_a [°C]", self.ambient_temperature_field),
            self.disturbance_enabled_checkbox,
            self._row("Amplitude [°C]", self.disturbance_amplitude_field),
            disturbance_times,
            widgets.Label("Affected Rooms"),
            self.affected_rooms_box,
            self._section("Simulation"),
            self._row("Stop Time [min]", self.stop_time_field),
            self.pacing_enabled_checkbox,
            widgets.Label("Pacing Rate [Sim s / Wall s]"),
            self.pacing_slider,
            widgets.HBox(
                [self.start_button, self.pause_button, self.stop_button],
                layout=widgets.Layout(justify_content="space-between"),
            ),
            widgets.HBox(
                [self.clear_button, self.save_metrics_button],
                layout=widgets.Layout(justify_content="space-between"),
            ),
            self.status_html,
            self.message_html,
        ]

        self.control_panel = widgets.VBox(
            left_children,
            layout=widgets.Layout(
                width="370px",
                min_width="370px",
                border="1px solid #bdbdbd",
                padding="10px",
                overflow="auto",
            ),
        )

        # RIGHT SIDE - TWO PLOTS
        self.plot_image = widgets.Image(
            format="png",
            layout=widgets.Layout(width="100%"),
        )
        self.plot_panel = widgets.VBox(
            [self.plot_image],
            layout=widgets.Layout(
                width="auto",
                min_width="0px",
                flex="1 1 auto",
                padding="6px",
            ),
        )

        self.root_widget = widgets.HBox(
            [self.control_panel, self.plot_panel],
            layout=widgets.Layout(
                width="100%",
                align_items="stretch",
                overflow="hidden",
            ),
        )

    def _load_configuration(self):
        self._ignore_callbacks = True
        try:
            refs = self.CFG.get("reference", {}).get("fourRooms", [22, 21, 20, 19])
            for i in range(4):
                self.reference_fields[i].value = float(refs[i])

            simulation = self.CFG.get("simulation", {})
            self.stop_time_field.value = float(simulation.get("stopTime", 1800.0)) / 60.0
            self.pacing_slider.value = float(simulation.get("pacing", 100.0))
            self.pacing_enabled_checkbox.value = bool(simulation.get("pacingEnabled", True))

            self.ambient_temperature_field.value = float(self.ENV.get("T_a", 8.0))
            disturbance = self.ENV.get("disturbance", {})
            self.disturbance_enabled_checkbox.value = bool(disturbance.get("enabled", False))
            self.disturbance_amplitude_field.value = float(disturbance.get("amplitude", 13.0))
            self.disturbance_start_field.value = float(disturbance.get("startTime", 5.0))
            self.disturbance_end_field.value = float(disturbance.get("endTime", 8.0))
            mask = disturbance.get("roomMask", [1, 1, 1, 1])
            for i in range(4):
                self.disturbance_room_checkboxes[i].value = bool(mask[i])
            self.disturbance_all_checkbox.value = all(bool(v) for v in mask[:4])
        finally:
            self._ignore_callbacks = False

        self._update_controller_presentation()
        self._update_disturbance_controls()
        self._update_pacing_controls()
        self._update_experiment_suggestion(force=True)

    def _apply_configuration(self):
        controller_id = int(self.controller_dropdown.value)

        # REFERENCES
        if controller_id == 4:
            self.CFG["reference"]["singleRoom"] = float(self.reference_fields[0].value)
        else:
            self.CFG["reference"]["fourRooms"] = [
                float(field.value) for field in self.reference_fields
            ]

        # SIMULATION
        if float(self.stop_time_field.value) <= 0:
            raise ValueError("Stop time must be greater than zero.")
        self.CFG["simulation"]["stopTime"] = float(self.stop_time_field.value) * 60.0
        self.CFG["simulation"]["pacing"] = float(self.pacing_slider.value)
        self.CFG["simulation"]["pacingEnabled"] = bool(self.pacing_enabled_checkbox.value)

        # ENVIRONMENT
        start_time = float(self.disturbance_start_field.value)
        end_time = float(self.disturbance_end_field.value)
        if self.disturbance_enabled_checkbox.value and end_time <= start_time:
            raise ValueError("Disturbance end time must be greater than start time.")

        self.ENV["T_a"] = float(self.ambient_temperature_field.value)
        self.ENV["disturbance"]["enabled"] = bool(self.disturbance_enabled_checkbox.value)
        self.ENV["disturbance"]["amplitude"] = float(self.disturbance_amplitude_field.value)
        self.ENV["disturbance"]["startTime"] = start_time
        self.ENV["disturbance"]["endTime"] = end_time

        if controller_id == 4:
            self.ENV["disturbance"]["roomMask"] = [
                int(self.disturbance_room_checkboxes[0].value), 0, 0, 0
            ]
        else:
            self.ENV["disturbance"]["roomMask"] = [
                int(box.value) for box in self.disturbance_room_checkboxes
            ]

        self.context["CONTROLLER_ID"] = controller_id
        return controller_id

    def _on_experiment_name_changed(self, change):
        if getattr(self, "_ignore_callbacks", False):
            return
        self._experiment_name_customized = True

    def _on_controller_changed(self, change):
        if getattr(self, "_ignore_callbacks", False):
            return
        self._update_controller_presentation()
        self._update_experiment_suggestion(force=False)
        self._clear_plot_data_only()
        self._render_plots()
        self._update_button_states()

    def _on_reference_changed(self, change):
        if getattr(self, "_ignore_callbacks", False):
            return
        try:
            self._apply_configuration()
            self._render_plots()
        except Exception as exc:
            self._show_message(str(exc), error=True)

    def _on_disturbance_enabled_changed(self, change):
        if getattr(self, "_ignore_callbacks", False):
            return
        self._update_disturbance_controls()
        self._update_experiment_suggestion(force=False)
        self._on_configuration_changed(change)

    def _on_stop_time_changed(self, change):
        if getattr(self, "_ignore_callbacks", False):
            return
        self._update_experiment_suggestion(force=False)
        self._on_configuration_changed(change)

    def _on_pacing_enabled_changed(self, change):
        if getattr(self, "_ignore_callbacks", False):
            return
        self._update_pacing_controls()
        self._on_configuration_changed(change)

    def _on_disturbance_room_changed(self, change):
        if getattr(self, "_ignore_callbacks", False):
            return
        self._update_all_rooms_state()
        self._on_configuration_changed(change)

    def _on_configuration_changed(self, change=None):
        if getattr(self, "_ignore_callbacks", False):
            return
        try:
            self._apply_configuration()
            if self._is_running():
                self._set_status("Changes apply to next run", "busy")
            else:
                self._set_status("Ready", "ready")
        except Exception as exc:
            self._set_status("Configuration error", "error")
            self._show_message(str(exc), error=True)

    def _is_running(self):
        if self._running_in_colab:
            return bool(self._colab_running)
        return self._simulation_thread is not None and self._simulation_thread.is_alive()

    def _start_simulation(self, _button=None):
        # ------------------------------------------------------------------
        # Google Colab mode
        # ------------------------------------------------------------------
        if self._running_in_colab:
            if self._is_running():
                return

            try:
                controller_id = self._apply_configuration()
                configure_logging(self.CFG)
            except Exception as exc:
                self._set_status("Configuration error", "error")
                self._show_message(str(exc), error=True)
                return

            self._clear_plot_data_only()
            self._last_run = None
            self._stop_event.clear()
            self._pause_event.clear()
            self._show_message("")

            # Freeze a run-specific copy of the configuration.
            run_cfg = copy.deepcopy(self.CFG)
            run_env = copy.deepcopy(self.ENV)
            run_sensor = copy.deepcopy(self.SENSOR)
            run_ctrl = copy.deepcopy(self.CTRL)
            run_model = copy.deepcopy(self.MODEL)
            experiment_name = str(self.experiment_name_field.value)

            # Colab mode intentionally ignores pacing and does not attempt
            # background GUI animation. The full experiment is calculated
            # first and the GUI is refreshed once when it finishes.
            self._colab_running = True
            self._set_status("Running...", "running")
            self._update_button_states()

            self._simulation_worker_colab(
                controller_id,
                run_cfg,
                run_model,
                run_env,
                run_sensor,
                run_ctrl,
                experiment_name,
            )
            return

        # ------------------------------------------------------------------
        # Local VS Code / Jupyter mode
        # Original threaded behavior with live plot updates.
        # ------------------------------------------------------------------
        if self._is_running():
            if self._pause_event.is_set():
                self._pause_event.clear()
                self._set_status("Running", "running")
                self._update_button_states()
            return

        try:
            controller_id = self._apply_configuration()
            configure_logging(self.CFG)
        except Exception as exc:
            self._set_status("Configuration error", "error")
            self._show_message(str(exc), error=True)
            return

        self._clear_plot_data_only()
        self._last_run = None
        self._stop_event.clear()
        self._pause_event.clear()
        self._set_status("Starting simulation...", "busy")
        self._show_message("")

        # Freeze a run-specific copy of the configuration. GUI edits during
        # the run are therefore applied to the next run, matching the project workflow.
        run_cfg = copy.deepcopy(self.CFG)
        run_env = copy.deepcopy(self.ENV)
        run_sensor = copy.deepcopy(self.SENSOR)
        run_ctrl = copy.deepcopy(self.CTRL)
        run_model = copy.deepcopy(self.MODEL)

        self._simulation_thread = threading.Thread(
            target=self._simulation_worker,
            args=(controller_id, run_cfg, run_model, run_env, run_sensor, run_ctrl),
            daemon=True,
        )
        self._simulation_thread.start()
        self._set_status("Running", "running")
        self._update_button_states()

    def _simulation_worker_colab(
        self,
        controller_id,
        CFG,
        MODEL,
        ENV,
        SENSOR,
        CTRL,
        experiment_name,
    ):
        """Run the complete experiment synchronously in Google Colab.

        Colab does not reliably repaint ipywidgets while a background worker
        is running. This mode therefore calculates the experiment without
        pacing or intermediate plot refreshes, then updates the complete GUI
        once at the end.
        """
        final_status = "Finished"
        final_state = "ready"
        final_message = ""
        final_message_is_error = False

        try:
            model = MaIA_SDCI_Temperature_Control(
                CFG, MODEL, ENV, SENSOR, CTRL, controller_id
            )
            stop_time = float(CFG["simulation"]["stopTime"])

            initial = model.current_sample()
            with self._data_lock:
                self._append_sample(initial)

            # No pacing and no intermediate GUI rendering in Colab.
            while model.time < stop_time - 1e-12:
                sample = model.step()
                with self._data_lock:
                    self._append_sample(sample)

            with self._data_lock:
                run_data = self._data_as_arrays()

            self._last_run = {
                "controllerID": controller_id,
                "experimentName": experiment_name,
                "data": run_data,
            }

        except ControllerInterfaceError as exc:
            final_status = "Controller implementation error"
            final_state = "error"
            final_message = str(exc)
            final_message_is_error = True

        except Exception as exc:
            final_status = "Simulation error"
            final_state = "error"
            final_message = (
                f"{exc}<br><pre style='white-space:pre-wrap'>"
                f"{traceback.format_exc()}</pre>"
            )
            final_message_is_error = True

        finally:
            self._colab_running = False

            # One final frontend update. This happens after the synchronous
            # callback has completed its computation, which is reliable in Colab.
            self._render_plots()
            self._set_status(final_status, final_state)

            if final_message:
                self._show_message(final_message, error=final_message_is_error)
            else:
                self._show_message("")

            self._update_button_states()

    def _simulation_worker(self, controller_id, CFG, MODEL, ENV, SENSOR, CTRL):
        stopped_by_user = False
        try:
            model = MaIA_SDCI_Temperature_Control(
                CFG, MODEL, ENV, SENSOR, CTRL, controller_id
            )
            stop_time = float(CFG["simulation"]["stopTime"])
            pacing_enabled = bool(CFG["simulation"].get("pacingEnabled", True))
            pacing_rate = max(float(CFG["simulation"].get("pacing", 100.0)), 1e-9)

            initial = model.current_sample()
            with self._data_lock:
                self._append_sample(initial)
            self._render_plots()

            while model.time < stop_time - 1e-12:
                if self._stop_event.is_set():
                    stopped_by_user = True
                    break

                while self._pause_event.is_set() and not self._stop_event.is_set():
                    time.sleep(0.05)
                if self._stop_event.is_set():
                    stopped_by_user = True
                    break

                wall_start = time.perf_counter()
                sample = model.step()
                with self._data_lock:
                    self._append_sample(sample)

                now = time.perf_counter()
                if now - self._last_render_wall_time >= 0.5 or model.time >= stop_time:
                    self._last_render_wall_time = now
                    self._render_plots()

                if pacing_enabled:
                    requested = 1.0 / pacing_rate
                    elapsed = time.perf_counter() - wall_start
                    if requested > elapsed:
                        time.sleep(requested - elapsed)

            with self._data_lock:
                run_data = self._data_as_arrays()
            self._last_run = {
                "controllerID": controller_id,
                "experimentName": self.experiment_name_field.value,
                "data": run_data,
            }
            self._render_plots()
            self._set_status("Stopped" if stopped_by_user else "Finished", "ready")
        except ControllerInterfaceError as exc:
            self._set_status("Controller implementation error", "error")
            self._show_message(str(exc), error=True)
        except Exception as exc:
            self._set_status("Simulation error", "error")
            self._show_message(
                f"{exc}<br><pre style='white-space:pre-wrap'>{traceback.format_exc()}</pre>",
                error=True,
            )
        finally:
            self._simulation_thread = None
            self._update_button_states()

    def _append_sample(self, sample):
        for key in self._latest_data:
            value = sample[key]
            if key == "time":
                self._latest_data[key].append(float(value))
            else:
                self._latest_data[key].append(np.asarray(value, dtype=float).copy())

    def _data_as_arrays(self):
        if not self._latest_data["time"]:
            return {
                "time": np.array([], dtype=float),
                "T_ref": np.empty((0, 4)),
                "T_meas": np.empty((0, 4)),
                "Q_heat": np.empty((0, 4)),
                "duty_cycle": np.empty((0, 4)),
                "T_room": np.empty((0, 4)),
                "T_dist": np.empty((0, 4)),
            }
        return {
            "time": np.asarray(self._latest_data["time"], dtype=float),
            **{
                key: np.vstack(self._latest_data[key])
                for key in ("T_ref", "T_meas", "Q_heat", "duty_cycle", "T_room", "T_dist")
            },
        }

    def _pause_simulation(self, _button=None):
        if self._running_in_colab:
            return
        if not self._is_running():
            return
        self._pause_event.set()
        self._set_status("Paused", "paused")
        self._update_button_states()

    def _stop_simulation(self, _button=None):
        if self._running_in_colab:
            return
        if self._is_running():
            self._stop_event.set()
            self._pause_event.clear()
            self._set_status("Stopping...", "busy")

    def _clear_plot_data_only(self):
        with self._data_lock:
            self._latest_data = self._empty_data()
        self._last_run = None
        self._plot_start_index = 0

    def _clear_plots(self, _button=None):
        running = self._is_running()
        paused = running and self._pause_event.is_set()

        if running:
            # Clear only the displayed history. Keep the complete run data so
            # final metrics still use the entire simulation.
            with self._data_lock:
                self._plot_start_index = len(self._latest_data["time"])
        else:
            self._clear_plot_data_only()

        self._render_plots()

        if paused:
            self._set_status("Paused", "paused")
        elif running:
            self._set_status("Running", "running")
        else:
            self._set_status("Ready", "ready")

    def _build_plot_figure(self):
        with self._data_lock:
            data = self._data_as_arrays()

        controller_id = int(self.controller_dropdown.value)
        active = [0] if controller_id == 4 else [0, 1, 2, 3]
        stop_minutes = max(float(self.stop_time_field.value), 0.001)

        fig = Figure(figsize=(10.0, 7.2), dpi=105, tight_layout=True)
        ax_t = fig.add_subplot(211)
        ax_q = fig.add_subplot(212)

        ax_t.set_title("Temperature Tracking", fontsize=16, fontweight="bold")
        ax_t.set_ylabel("Temperature [°C]", fontsize=12, fontweight="bold")
        ax_t.grid(True)
        ax_q.set_title("Heater Thermal Power", fontsize=16, fontweight="bold")
        ax_q.set_xlabel("Time [min]", fontsize=12, fontweight="bold")
        ax_q.set_ylabel("Thermal Power [W]", fontsize=12, fontweight="bold")
        ax_q.grid(True)

        display_start = min(int(self._plot_start_index), int(data["time"].size))

        if data["time"].size > display_start:
            # Decimate only for display. Clearing plots during a run changes
            # the visible history but preserves complete data for metrics.
            display_indices = np.arange(display_start, data["time"].size)
            if display_indices.size > self.MaxPlotPoints:
                display_indices = np.unique(
                    np.round(
                        np.linspace(
                            display_indices[0],
                            display_indices[-1],
                            self.MaxPlotPoints,
                        )
                    ).astype(int)
                )
            indices = display_indices
            t_min = data["time"][indices] / 60.0
            current_minutes = float(data["time"][-1] / 60.0)
            plot_end = max(stop_minutes, current_minutes, 1.0)

            for room in active:
                if controller_id == 4:
                    room_number = 5
                else:
                    room_number = room + 1
                ax_t.plot(
                    t_min,
                    data["T_ref"][indices, room],
                    "--",
                    linewidth=0.9,
                    label=f"Tref{room_number}",
                )
                ax_t.plot(
                    t_min,
                    data["T_meas"][indices, room],
                    "-",
                    linewidth=1.5,
                    label=f"T{room_number}",
                )
                ax_q.plot(
                    t_min,
                    data["Q_heat"][indices, room],
                    "-",
                    linewidth=1.2,
                    label=f"Q{room_number}",
                )
        else:
            plot_end = max(stop_minutes, 1.0)
            if controller_id == 4:
                reference_values = [float(self.reference_fields[0].value)]
                room_numbers = [5]
            else:
                reference_values = [float(self.reference_fields[i].value) for i in active]
                room_numbers = [i + 1 for i in active]
            for value, room_number in zip(reference_values, room_numbers):
                ax_t.plot([0, plot_end], [value, value], "--", linewidth=0.9, label=f"Tref{room_number}")

        ax_t.set_xlim(0, plot_end)
        ax_q.set_xlim(0, plot_end)
        ax_t.legend(loc="lower right", ncol=min(4, max(1, 2 * len(active))), fontsize=8)
        ax_q.legend(loc="lower right", ncol=min(4, max(1, len(active))), fontsize=8)

        return fig

    def _render_plots(self):
        try:
            fig = self._build_plot_figure()
            buffer = BytesIO()
            fig.savefig(buffer, format="png", dpi=110, bbox_inches="tight")
            self.plot_image.value = buffer.getvalue()
        except Exception as exc:
            self._show_message(f"Plot update error: {exc}", error=True)

    def _build_gui_snapshot_figure(self):
        """Build a portable PNG representation of the complete current GUI."""

        # Render the current two-panel plot first.
        plot_figure = self._build_plot_figure()
        plot_buffer = BytesIO()
        plot_figure.savefig(plot_buffer, format="png", dpi=120, bbox_inches="tight")
        plot_buffer.seek(0)
        plot_image = mpl_image.imread(plot_buffer, format="png")

        # Match the wide layout used by the notebook GUI.
        figure = Figure(figsize=(14.8, 8.2), dpi=110)
        left = figure.add_axes([0.008, 0.018, 0.305, 0.964])
        right = figure.add_axes([0.318, 0.018, 0.677, 0.964])

        # RIGHT SIDE - current plots
        right.imshow(plot_image)
        right.axis("off")

        # LEFT SIDE - current controls
        left.set_xlim(0.0, 1.0)
        left.set_ylim(0.0, 1.0)
        left.axis("off")
        left.add_patch(
            Rectangle(
                (0.0, 0.0),
                1.0,
                1.0,
                facecolor="#FFFFFF",
                edgecolor="#BDBDBD",
                linewidth=1.0,
            )
        )

        def text(x, y, value, size=9.5, weight="normal", color="#222222",
                 ha="left", va="center"):
            left.text(
                x, y, str(value),
                fontsize=size,
                fontweight=weight,
                color=color,
                ha=ha,
                va=va,
                family="sans-serif",
            )

        def field(x, y, width, value, disabled=False):
            height = 0.043
            face = "#F4F4F4" if disabled else "#FFFFFF"
            edge = "#C7C7C7"
            color = "#8A8A8A" if disabled else "#444444"
            left.add_patch(
                Rectangle(
                    (x, y - height / 2),
                    width,
                    height,
                    facecolor=face,
                    edgecolor=edge,
                    linewidth=0.8,
                )
            )
            text(x + 0.018, y, value, size=9.4, color=color)

        def checkbox(x, y, checked, label, disabled=False):
            size = 0.027
            face = "#D0D0D0" if disabled else "#FFFFFF"
            edge = "#C5C5C5" if disabled else "#777777"
            color = "#FFFFFF" if disabled else "#555555"
            left.add_patch(
                Rectangle(
                    (x, y - size / 2),
                    size,
                    size,
                    facecolor=face,
                    edgecolor=edge,
                    linewidth=0.8,
                )
            )
            if checked:
                text(x + size / 2, y, "✓", size=10.5, weight="bold",
                     color=color, ha="center")
            text(x + size + 0.018, y, label, size=9.2,
                 color="#777777" if disabled else "#222222")

        def button(x, y, width, label, color, disabled=False):
            height = 0.045
            face = "#EFEFEF" if disabled else color
            label_color = "#999999" if disabled else (
                "#FFFFFF" if color != "#EFEFEF" else "#333333"
            )
            left.add_patch(
                Rectangle(
                    (x, y - height / 2),
                    width,
                    height,
                    facecolor=face,
                    edgecolor="#FFFFFF",
                    linewidth=0.8,
                )
            )
            text(
                x + width / 2,
                y,
                label,
                size=9.5,
                color=label_color,
                ha="center",
            )

        def option_label(widget):
            for option in widget.options:
                if isinstance(option, (tuple, list)) and len(option) == 2:
                    if option[1] == widget.value:
                        return option[0]
                elif option == widget.value:
                    return option
            return widget.value

        # Setup
        text(0.035, 0.958, "Experiment")
        field(0.395, 0.958, 0.565, self.experiment_name_field.value)

        text(0.035, 0.908, "Controller")
        field(0.395, 0.908, 0.565, option_label(self.controller_dropdown))

        text(0.035, 0.858, "Model Type")
        field(0.395, 0.858, 0.565, option_label(self.model_type_dropdown), disabled=True)

        controller_id = int(self.controller_dropdown.value)

        # References
        if controller_id == 4:
            text(0.035, 0.808, "ESC Reference [°C]", weight="bold")
            text(0.035, 0.757, "Room 5", size=9.0)
            field(0.035, 0.715, 0.195, f"{float(self.reference_fields[0].value):g}")
        else:
            text(0.035, 0.808, "Temperature References [°C]", weight="bold")
            x_positions = [0.035, 0.280, 0.525, 0.770]
            for i, x in enumerate(x_positions):
                text(x, 0.757, f"Room {i + 1}", size=9.0)
                field(x, 0.715, 0.195, f"{float(self.reference_fields[i].value):g}")

        # Environment
        text(0.035, 0.657, "Environment", size=11.0, weight="bold")
        text(0.035, 0.607, "T_a [°C]")
        field(0.395, 0.607, 0.565, f"{float(self.ambient_temperature_field.value):g}")

        disturbance_enabled = bool(self.disturbance_enabled_checkbox.value)
        checkbox(
            0.035, 0.558,
            disturbance_enabled,
            "Enable temperature disturbance",
            disabled=False,
        )

        text(0.035, 0.508, "Amplitude [°C]")
        field(
            0.395, 0.508, 0.565,
            f"{float(self.disturbance_amplitude_field.value):g}",
            disabled=not disturbance_enabled,
        )

        text(0.035, 0.458, "Start [min]")
        field(
            0.240, 0.458, 0.215,
            f"{float(self.disturbance_start_field.value):g}",
            disabled=not disturbance_enabled,
        )
        text(0.475, 0.458, "End [min]")
        field(
            0.665, 0.458, 0.215,
            f"{float(self.disturbance_end_field.value):g}",
            disabled=not disturbance_enabled,
        )

        text(0.035, 0.407, "Affected Rooms")
        if controller_id == 4:
            checkbox(
                0.035, 0.365,
                bool(self.disturbance_room_checkboxes[0].value),
                "5",
                disabled=not disturbance_enabled,
            )
        else:
            checkbox(
                0.035, 0.365,
                bool(self.disturbance_all_checkbox.value),
                "All",
                disabled=not disturbance_enabled,
            )
            for i, x in enumerate([0.205, 0.355, 0.505, 0.655]):
                checkbox(
                    x, 0.365,
                    bool(self.disturbance_room_checkboxes[i].value),
                    str(i + 1),
                    disabled=not disturbance_enabled,
                )

        # Simulation
        text(0.035, 0.310, "Simulation", size=11.0, weight="bold")
        text(0.035, 0.260, "Stop Time [min]")
        field(0.395, 0.260, 0.565, f"{float(self.stop_time_field.value):g}")

        pacing_enabled = bool(self.pacing_enabled_checkbox.value)
        checkbox(
            0.035, 0.211,
            pacing_enabled,
            (
                "Enable simulation pacing (local only)"
                if self._running_in_colab
                else "Enable simulation pacing"
            ),
            disabled=self._running_in_colab,
        )

        text(0.035, 0.161, "Pacing Rate [Sim s / Wall s]")
        slider_x0 = 0.055
        slider_x1 = 0.700
        slider_y = 0.118
        left.plot(
            [slider_x0, slider_x1],
            [slider_y, slider_y],
            color="#D6D6D6",
            linewidth=3.0,
            solid_capstyle="round",
        )
        slider_min = float(self.pacing_slider.min)
        slider_max = float(self.pacing_slider.max)
        slider_value = float(self.pacing_slider.value)
        fraction = (slider_value - slider_min) / max(slider_max - slider_min, 1e-12)
        knob_x = slider_x0 + np.clip(fraction, 0.0, 1.0) * (slider_x1 - slider_x0)
        left.plot(
            knob_x,
            slider_y,
            marker="o",
            markersize=9,
            markerfacecolor="#FFFFFF" if pacing_enabled else "#F2F2F2",
            markeredgecolor="#CFCFCF",
        )
        text(0.870, slider_y, f"{slider_value:.2f}", size=9.2, ha="right")

        running = self._is_running()
        paused = running and self._pause_event.is_set()

        if self._running_in_colab:
            button(0.035, 0.074, 0.300, "START", "#4CAF50",
                   disabled=running)
            button(0.345, 0.074, 0.300, "PAUSE", "#F7B955",
                   disabled=True)
            button(0.655, 0.074, 0.300, "STOP", "#F18680",
                   disabled=True)
        else:
            button(0.035, 0.074, 0.300, "START", "#4CAF50",
                   disabled=running and not paused)
            button(0.345, 0.074, 0.300, "PAUSE", "#F7B955",
                   disabled=(not running) or paused)
            button(0.655, 0.074, 0.300, "STOP", "#F18680",
                   disabled=not running)

        button(0.035, 0.028, 0.405, "Clear Plots", "#EFEFEF", disabled=False)
        button(0.550, 0.028, 0.405, "Save Metrics", "#EFEFEF", disabled=False)

        return figure

    def _save_metrics(self, _button=None):
        if self._is_running():
            self._show_message("Stop or complete the simulation before saving metrics.", error=True)
            return
        if self._last_run is None:
            self._show_message("No completed simulation is available.", error=True)
            return

        try:
            run = self._last_run
            data = run["data"]
            metrics = compute_metrics(
                data["time"],
                data["T_ref"],
                data["T_meas"],
                data["time"],
                data["Q_heat"],
                run["controllerID"],
            )

            # Save the complete current GUI view, not only the two plots.
            self._set_status("Metrics saved", "ready")
            gui_figure = self._build_gui_snapshot_figure()
            save_results(
                metrics,
                gui_figure,
                self.experiment_name_field.value,
                self.PROJECT["resultsFolder"],
            )
            self._show_message("")
        except Exception as exc:
            self._set_status("Save error", "error")
            self._show_message(str(exc), error=True)

    def _update_experiment_suggestion(self, force=False):
        if not force and self._experiment_name_customized:
            return
        controller_name = self.CONTROLLER_NAMES[int(self.controller_dropdown.value)]
        disturbance_name = "DIST" if self.disturbance_enabled_checkbox.value else "NOM"
        stop_minutes = float(self.stop_time_field.value)
        if abs(stop_minutes - round(stop_minutes)) < 1e-9:
            stop_text = f"{int(round(stop_minutes))}min"
        else:
            stop_text = f"{stop_minutes:.2f}".rstrip("0").rstrip(".").replace(".", "p") + "min"
        self._ignore_callbacks = True
        self.experiment_name_field.value = f"{controller_name}_EQUATION_{disturbance_name}_{stop_text}"
        self._ignore_callbacks = False
        self._experiment_name_customized = False

    def _update_controller_presentation(self):
        controller_id = int(self.controller_dropdown.value)
        self._ignore_callbacks = True
        try:
            if controller_id == 4:
                self.reference_title.value = "<b>ESC Reference [°C]</b>"
                self.reference_room_labels[0].value = "Room 5"
                self.reference_fields[0].value = float(self.CFG["reference"].get("singleRoom", 22.0))
                self.reference_boxes[0].layout.display = "flex"
                for i in range(1, 4):
                    self.reference_boxes[i].layout.display = "none"

                self.disturbance_all_checkbox.layout.display = "none"
                self.disturbance_room_checkboxes[0].description = "5"
                self.disturbance_room_checkboxes[0].layout.display = "flex"
                for i in range(1, 4):
                    self.disturbance_room_checkboxes[i].layout.display = "none"
            else:
                self.reference_title.value = "<b>Temperature References [°C]</b>"
                refs = self.CFG["reference"].get("fourRooms", [22, 21, 20, 19])
                for i in range(4):
                    self.reference_room_labels[i].value = f"Room {i + 1}"
                    self.reference_fields[i].value = float(refs[i])
                    self.reference_boxes[i].layout.display = "flex"

                self.disturbance_all_checkbox.layout.display = "flex"
                for i in range(4):
                    self.disturbance_room_checkboxes[i].description = str(i + 1)
                    self.disturbance_room_checkboxes[i].layout.display = "flex"
        finally:
            self._ignore_callbacks = False
        self._update_disturbance_controls()

    def _update_button_states(self):
        running = self._is_running()

        if self._running_in_colab:
            self.start_button.disabled = running
            self.pause_button.disabled = True
            self.stop_button.disabled = True
        else:
            paused = running and self._pause_event.is_set()
            self.start_button.disabled = running and not paused
            self.pause_button.disabled = (not running) or paused
            self.stop_button.disabled = not running

        # Keep result controls available at all times.
        self.clear_button.disabled = False
        self.save_metrics_button.disabled = False

    def _set_status(self, text, state="ready"):
        self._status_text = str(text)
        self._status_state = str(state)

        colors = {
            "ready": "#777777",
            "busy": "#F39C12",
            "running": "#2EAF57",
            "paused": "#E0A000",
            "error": "#D64040",
        }
        color = colors.get(state, colors["ready"])
        self.status_html.value = (
            f"<b>Status</b>&nbsp;&nbsp;"
            f"<span style='display:inline-block;width:12px;height:12px;border-radius:50%;background:{color};'></span>"
            f"&nbsp;&nbsp;<b>{text}</b>"
        )
        self._update_button_states()

    def _show_message(self, text, error=False):
        if not text:
            self.message_html.value = ""
            return
        color = "#B00020" if error else "#333333"
        self.message_html.value = f"<div style='color:{color};font-size:11px'>{text}</div>"

    def _update_disturbance_controls(self):
        enabled = bool(self.disturbance_enabled_checkbox.value)
        self.disturbance_amplitude_field.disabled = not enabled
        self.disturbance_start_field.disabled = not enabled
        self.disturbance_end_field.disabled = not enabled
        self.disturbance_all_checkbox.disabled = not enabled
        for box in self.disturbance_room_checkboxes:
            box.disabled = not enabled

    def _set_all_disturbance_rooms(self, change):
        if getattr(self, "_ignore_callbacks", False):
            return
        self._ignore_callbacks = True
        try:
            for box in self.disturbance_room_checkboxes:
                box.value = bool(change["new"])
        finally:
            self._ignore_callbacks = False
        self._on_configuration_changed()

    def _update_all_rooms_state(self):
        if int(self.controller_dropdown.value) == 4:
            return
        self._ignore_callbacks = True
        try:
            self.disturbance_all_checkbox.value = all(
                box.value for box in self.disturbance_room_checkboxes
            )
        finally:
            self._ignore_callbacks = False

    def _update_pacing_controls(self):
        if self._running_in_colab:
            self.pacing_enabled_checkbox.disabled = True
            self.pacing_slider.disabled = True
            return

        self.pacing_enabled_checkbox.disabled = False
        self.pacing_slider.disabled = not bool(self.pacing_enabled_checkbox.value)

    def display(self):
        self._update_button_states()
        display(self.root_widget)
        return self.root_widget
